# RanPAC Phase 2 · Ablation và Thay thế hàm kích hoạt sau Random Projection ($\phi$)

Notebook này thuộc bản **tái cài đặt từ mô tả bài báo (Re-implementation)** phục vụ phần **Critical Analysis & Empirical Demonstration** (*Component Ablations* và *Replacement Baselines*) trên **Split CIFAR-100** (seed `1993`, 10 task × 10 lớp, ViT-B/16 ImageNet-21K đóng băng, không Phase 1).

### Động lực lý thuyết
Bài báo lập luận (Section 4.3, Property 2) rằng phép chiếu ngẫu nhiên $H = \phi(F W)$ cần hàm phi tuyến $\phi = \operatorname{ReLU}$ để tạo tương tác bậc cao giữa các chiều đặc trưng và nâng hạng của ma trận Gram $G = H^\top H$ vượt quá chiều gốc $L = 768$. Tuy nhiên, $\operatorname{ReLU}(z) = \max(0, z)$ trộn lẫn bốn đặc tính: (1) phi tuyến, (2) thưa (~50% phần tử bằng 0), (3) lệch dương ($\mathbb{E}[\phi(z)] > 0$), và (4) tuyến tính từng khúc không bị chặn.

Để tách bạch đóng góp thực sự của từng đặc tính, notebook giữ cố định $F$, cùng ma trận $W \sim \mathcal{N}(0, 1)$ (seed `1993`), cùng quy tắc chọn $\lambda_t$ (80/20 CV) và đối chiếu 6 hàm $\phi$ tại $M \in \{768, 2500, 10000\}$:

| Mã `activation` | Công thức $\phi(z)$ | Đặc tính cần kiểm chứng |
| --- | --- | --- |
| `linear` | $z$ | Đối chứng tuyến tính: kiểm chứng Property 2 ($\operatorname{rank}(G) \le 768$, $A[10]$ đi ngang quanh ~87%). |
| `relu` | $\max(0, z)$ | Thiết lập gốc của RanPAC: thưa ~50%, lệch dương ($\lambda_1 / \lambda_2 \gg 1$). |
| `leaky_relu` | $\max(0.1z, z)$ | Giữ điểm gãy phi tuyến tại 0 nhưng **xóa tính thưa** (sparsity = 0%). |
| `gelu` | $z \Phi(z)$ | Hàm phi tuyến trơn chuẩn của ViT, xấp xỉ mềm của ReLU. |
| `tanh` | $\tanh(z)$ | Hàm lẻ đối xứng tâm ($\mathbb{E}[\phi(z)] \approx 0$), bị chặn trong $(-1, 1)$ (kiểm tra bão hòa biên). |
| `abs` | $|z|$ | Hàm chẵn ($\phi(-z) = \phi(z)$): xóa hoàn toàn thông tin dấu tuyến tính bậc 1, chỉ giữ tương tác bậc chẵn. |

Đồng thời, notebook dùng `gram_spectrum_stats` để giải thích hiện tượng **trị riêng trung bình dương $\lambda_1$ làm lệch phép tính `matrix_rank` chuẩn trên FP32** (từng quan sát thấy ở `sweep_rank_g.csv` khi $M=10000$).


## 1. Cài đặt thư viện và mở mã dùng chung

Đặt `ranpac_core.py`, `ranpac_data.py`, `ranpac_colab.py`, `ranpac_full.py` và `test_ranpac_core.py` trong `MyDrive/RanPAC_implement`. Dùng chung `WORK_DIR` để tái sử dụng trực tiếp cache embedding ViT-B/16. Mọi run mới log vào project W&B công khai `pnp123/RanPAC-reproduction` với tag `['empirics', 'activation-ablation']`.


In [ ]:
%pip install -q timm==1.0.22 wandb==0.31.0


In [ ]:
import gc
import sys
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')
SOURCE_DIR = Path('/content/drive/MyDrive/RanPAC_implement')
WORK_DIR = Path('/content/drive/MyDrive/ranpac-runs')
DATA_ROOT = '/content/ranpac-data'

if not (SOURCE_DIR / 'ranpac_colab.py').is_file():
    raise FileNotFoundError(f'Chưa tìm thấy mã nguồn tại {SOURCE_DIR}')
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from ranpac_colab import configure_wandb, run_phase2
from ranpac_core import ACTIVATIONS, NUM_TASKS, SEED, Ridge, gram_spectrum_stats, split_class_order
from ranpac_data import FEATURE_DIM, FeatureCache, atomic_output, feature_config


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromName('test_ranpac_core')
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError('Kiểm tra unit test trên CPU chưa đạt.')


## 2. Cấu hình lưới thí nghiệm ($\phi \times M$) và chuẩn bị Feature Cache

- Mặc định chạy đủ 6 hàm kích hoạt ở `M = 2500` và `M = 10000` (cộng thêm mốc kiểm tra ma trận vuông `M = 768` cho nhóm cốt lõi hoặc toàn bộ 6 hàm tùy thời gian).
- Để rút ngắn thời gian chạy nếu chỉ cần bảng chính cho Poster, có thể thu gọn `SELECTED_ACTIVATIONS = ('linear', 'relu', 'leaky_relu', 'gelu', 'tanh')` hoặc `SELECTED_DIMS = (768, 2500, 10000)`.
- Nếu trên Drive đã có sẵn checkpoint hoàn tất 10 task của `relu` (từ Notebook 03/05) hoặc `linear` (từ Notebook 07), bật `REUSE_EXISTING_RUNS = True` để dùng lại trực tiếp không cần giải lại.


In [ ]:
WANDB_ENTITY = 'pnp123'
WANDB_PROJECT = 'RanPAC-reproduction'
WANDB_MODE = 'online'
RUN_LABEL = 'act-ablation-01'

BATCH_SIZE = 48
NUM_WORKERS = 2
STATS_DEVICE = 'cpu'
RIDGE_NO_RP_A10_PCT = 87.02

SELECTED_ACTIVATIONS = ('linear', 'relu', 'leaky_relu', 'gelu', 'tanh', 'abs')
SELECTED_DIMS = (768, 2500, 10000)
REUSE_EXISTING_RUNS = True

EXISTING_RUN_MAP = {
    ('relu', 768): WORK_DIR / 'runs' / f'rp-act-relu-m768-s{SEED}-linear-sweep-01',
    ('relu', 2500): WORK_DIR / 'runs' / f'rp-sweep-m2500-s{SEED}-m-sweep-01',
    ('relu', 10000): WORK_DIR / 'runs' / f'rp-table1-s{SEED}-main',
    ('linear', 768): WORK_DIR / 'runs' / f'rp-linear-m768-s{SEED}-linear-sweep-01',
    ('linear', 2500): WORK_DIR / 'runs' / f'rp-linear-m2500-s{SEED}-linear-sweep-01',
    ('linear', 10000): WORK_DIR / 'runs' / f'rp-linear-m10000-s{SEED}-linear-sweep-01',
}

for act in SELECTED_ACTIVATIONS:
    if act not in ACTIVATIONS:
        raise ValueError(f'Activation không hợp lệ: {act}')

if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime trên Colab trước khi chạy.')
if WANDB_MODE == 'online':
    configure_wandb(WANDB_ENTITY, WANDB_MODE)

feature_device = torch.device('cuda')
cache = FeatureCache(
    WORK_DIR / 'embeddings', DATA_ROOT, feature_config(split_class_order(), feature_device),
    feature_device, BATCH_SIZE, NUM_WORKERS,
)
for task in range(1, NUM_TASKS + 1):
    for split in ('train', 'test'):
        shard = cache.root / f'{split}-task-{task:02}.pt'
        if not shard.is_file():
            print(f'Trích embedding {split} cho task {task:02}...', flush=True)
            cache.load(split, task)
sample_test_features, _, _ = cache.load('test', 1)
print(f'Feature cache sẵn sàng tại: {cache.root}')
del cache
gc.collect()
torch.cuda.empty_cache()


## 3. Chạy sweep $(\phi, M)$ và chẩn đoán phổ trị riêng của ma trận Gram $G$

Với mỗi cặp $(\phi, M)$ hoàn tất đủ 10 task, chương trình trích từ `checkpoint.pt`:
- `a10_pct`: độ chính xác trung bình cuối cùng sau 10 task.
- `sparsity_pct`: tỷ lệ phần tử đúng bằng `0` của $H = \phi(F W)$ trên shard test mẫu.
- `rank_standard`: số trị riêng vượt ngưỡng $\lambda_1 \cdot M \cdot \epsilon_{\text{fp32}}$.
- `rank_deflated_dc`: hạng số học sau khi tách trị riêng trung bình dương $\lambda_1$ (dùng ngưỡng $\lambda_2 \cdot M \cdot \epsilon_{\text{fp32}}$), phản ánh đúng số chiều thông tin thực sự của ma trận Gram tích lũy ở FP32.
- `effective_rank` và tỷ số `top1_to_top2_ratio` ($\lambda_1 / \lambda_2$).


In [ ]:
@torch.no_grad()
def measure_sparsity_pct(features_cpu, dim, activation):
    probe = Ridge(FEATURE_DIM, projection_dim=dim, seed=SEED, activation=activation)
    projected = probe.project(features_cpu.clone())
    return 100.0 * float((projected == 0).float().mean().item())


def run_or_load_pair(act, dim):
    if REUSE_EXISTING_RUNS and (act, dim) in EXISTING_RUN_MAP:
        cand_dir = EXISTING_RUN_MAP[(act, dim)]
        cand_ckpt = cand_dir / 'checkpoint.pt'
        if cand_ckpt.is_file():
            saved = torch.load(cand_ckpt, map_location='cpu', weights_only=True)
            if len(saved.get('records', ())) == NUM_TASKS and saved.get('model') is not None:
                print(f'[Tái sử dụng] ({act}, M={dim}) từ {cand_dir.name}')
                return saved, cand_dir

    preset = f'rp-act-{act}-m{dim}'
    run_dir = WORK_DIR / 'runs' / f'{preset}-s{SEED}-{RUN_LABEL}'
    ckpt_path = run_dir / 'checkpoint.pt'
    resume_flag = False
    if ckpt_path.is_file():
        saved = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        if len(saved.get('records', ())) == NUM_TASKS and saved.get('model') is not None:
            print(f'[Đã xong] {run_dir.name} -> A[10] = {saved["records"][-1]["avg_accuracy_pct"]:.2f}%')
            return saved, run_dir
        resume_flag = True

    run_phase2(
        preset, WORK_DIR, entity=WANDB_ENTITY, project=WANDB_PROJECT,
        stop_after_task=NUM_TASKS, run_label=RUN_LABEL, resume=resume_flag,
        wandb_mode=WANDB_MODE, batch_size=BATCH_SIZE, workers=NUM_WORKERS,
        stats_device=STATS_DEVICE, measure_rank=True,
        extra_tags=['empirics', 'activation-ablation', f'act-{act}'],
        wandb_group_override=f'cifar100-activation-ablation-{RUN_LABEL}',
    )
    return torch.load(ckpt_path, map_location='cpu', weights_only=True), run_dir


rows = []
curves = []
for act in SELECTED_ACTIVATIONS:
    for dim in SELECTED_DIMS:
        ckpt, run_dir = run_or_load_pair(act, dim)
        records = ckpt['records']
        gram = ckpt['model']['G']
        spec = gram_spectrum_stats(gram)
        sparsity = measure_sparsity_pct(sample_test_features, dim, act)
        fit_eval_s = sum(r['fit_seconds'] + r['eval_seconds'] for r in records)
        solve_s = sum(r['ridge_solve_seconds'] or 0.0 for r in records)
        rows.append({
            'activation': act,
            'M': dim,
            'a10_pct': records[-1]['avg_accuracy_pct'],
            'delta_vs_ridge_pp': records[-1]['avg_accuracy_pct'] - RIDGE_NO_RP_A10_PCT,
            'sparsity_pct': sparsity,
            'rank_standard_fp32': spec['rank_standard'],
            'rank_deflated_dc': spec['rank_deflated_dc'],
            'effective_rank': spec['effective_rank'],
            'top1_to_top2_ratio': spec['top1_to_top2_ratio'],
            'fit_eval_seconds': fit_eval_s,
            'ridge_solve_seconds': solve_s,
            'wandb_url': ckpt.get('wandb', {}).get('url'),
        })
        for r in records:
            curves.append({
                'activation': act,
                'M': dim,
                'task': r['task'],
                'avg_accuracy_pct': r['avg_accuracy_pct'],
                'ridge_lambda': r['ridge_lambda'],
            })
        del ckpt, gram
        gc.collect()

df_act = pd.DataFrame(rows)
df_curves = pd.DataFrame(curves)
summary_dir = WORK_DIR / 'sweeps' / f'activation-ablation-s{SEED}-{RUN_LABEL}'
with atomic_output(summary_dir / 'activation_ablation_summary.csv') as tmp:
    df_act.to_csv(tmp, index=False)
with atomic_output(summary_dir / 'activation_task_curves.csv') as tmp:
    df_curves.to_csv(tmp, index=False)

display(df_act.round(3))


## 4. Trực quan hóa kết quả cho Slide / Poster

Biểu đồ ba cột đối chiếu đồng thời:
1. **$A[10]$ theo $M$** cho từng hàm $\phi$ (so với đường cơ sở Ridge không RP `87.02%`).
2. **Hạng số học đã khử lệch tâm (`rank_deflated_dc`)** theo $M$ (đường `linear` bị khóa ở $\le 768$, trong khi các hàm phi tuyến mở rộng không gian đặc trưng tuyến tính theo $M$).
3. **So sánh `rank_standard` vs `rank_deflated_dc` ở mức $M$ lớn nhất**, minh họa trực quan sai số ngưỡng FP32 do trị riêng $\lambda_1$ của các hàm lệch dương (`relu`, `gelu`, `abs`).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5), layout='constrained')
markers = {'linear': 's', 'relu': 'o', 'leaky_relu': '^', 'gelu': 'D', 'tanh': 'v', 'abs': 'P'}

for act in SELECTED_ACTIVATIONS:
    sub = df_act[df_act['activation'] == act].sort_values('M')
    mk = markers.get(act, 'o')
    axes[0].plot(sub['M'], sub['a10_pct'], f'{mk}-', label=act)
    axes[1].plot(sub['M'], sub['rank_deflated_dc'], f'{mk}-', label=act)

axes[0].axhline(RIDGE_NO_RP_A10_PCT, linestyle='--', color='gray', label='Ridge không RP (87.02%)')
axes[0].set_xscale('log')
axes[0].set_xticks(SELECTED_DIMS, labels=[str(d) for d in SELECTED_DIMS])
axes[0].minorticks_off()
axes[0].set_xlabel('Kích thước projection M')
axes[0].set_ylabel('A[10] (%)')
axes[0].set_title('Độ chính xác A[10] theo hàm kích hoạt φ')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.25)

axes[1].axhline(FEATURE_DIM, linestyle='--', color='tab:red', label=f'Giới hạn tuyến tính L={FEATURE_DIM}')
axes[1].set_xscale('log')
axes[1].set_xticks(SELECTED_DIMS, labels=[str(d) for d in SELECTED_DIMS])
axes[1].minorticks_off()
axes[1].set_xlabel('Kích thước projection M')
axes[1].set_ylabel('Rank(G) sau khi tách DC spike')
axes[1].set_title('Mở rộng hạng không gian đặc trưng theo φ')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)

max_dim = max(SELECTED_DIMS)
sub_max = df_act[df_act['M'] == max_dim].set_index('activation').reindex(SELECTED_ACTIVATIONS)
x = range(len(sub_max))
w = 0.36
axes[2].bar([i - w / 2 for i in x], sub_max['rank_standard_fp32'], width=w, label='rank_standard (λ1·M·eps)')
axes[2].bar([i + w / 2 for i in x], sub_max['rank_deflated_dc'], width=w, label='rank_deflated_dc (λ2·M·eps)')
axes[2].axhline(FEATURE_DIM, linestyle='--', color='tab:red', alpha=0.7)
axes[2].set_xticks(list(x), labels=list(sub_max.index), rotation=20)
axes[2].set_ylabel(f'Numerical rank tại M={max_dim}')
axes[2].set_title(f'Ảnh hưởng của DC eigenvalue λ1 tại M={max_dim}')
axes[2].legend(fontsize=8)
axes[2].grid(axis='y', alpha=0.25)

with atomic_output(summary_dir / 'activation_ablation_plots.png') as tmp:
    fig.savefig(tmp, format='png', dpi=150)
plt.show()
print(f'Đã lưu kết quả và hình vẽ tại: {summary_dir}')
